In [ ]:
from pathlib import Path

for root in Path("/kaggle/input").iterdir():
    print(root)

In [ ]:
from pathlib import Path

CODE_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/rsna-knee-abnormality-code"
)

for path in CODE_ROOT.rglob("*"):
    print(path)

In [ ]:
import shutil
from pathlib import Path

CODE_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/rsna-knee-abnormality-code"
)

WORK_ROOT = Path("/kaggle/working/rsna-knee-abnormality-code")

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

shutil.copytree(CODE_ROOT, WORK_ROOT)

print("Code copied to:")
print(WORK_ROOT)
print("\nProject folders:")
for p in WORK_ROOT.iterdir():
    if p.is_dir():
        print(" -", p.name)

In [ ]:
import sys

sys.path.insert(
    0,
    "/kaggle/working/rsna-knee-abnormality-code"
)

from src.data.dataset import TARGETS
from src.models.dinov2 import DINOv2Backbone

print("Import successful!")
print("Targets:", TARGETS)


In [ ]:
import pandas as pd
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

train = pd.read_csv(DATA_ROOT / "train.csv")
train_series = pd.read_csv(DATA_ROOT / "train_series.csv")

print("Train shape:", train.shape)
print("Train series shape:", train_series.shape)

print("\nLabeled studies:", train["ACL"].notna().sum())
print("Total studies:", len(train))

print("\nSeries columns:")
print(train_series.columns.tolist())

In [ ]:
import pandas as pd
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

train = pd.read_csv(DATA_ROOT / "train.csv")
series = pd.read_csv(DATA_ROOT / "train_series.csv")

# Pick the first labeled study
study_uid = train.loc[train["ACL"].notna(), "StudyInstanceUID"].iloc[0]

study_series = series[
    (series["StudyInstanceUID"] == study_uid)
    & (series["Fluid_Sensitive"] == 1)
    & (series["Anatomical_Plane"].isin(["Sagittal", "Coronal", "Axial"]))
]

print("Study:", study_uid)
print("\nFluid-sensitive series:")
print(
    study_series[
        ["SeriesInstanceUID", "Anatomical_Plane"]
    ].to_string(index=False)
)

print("\nNumber of series:", len(study_series))

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/rsna-knee-abnormality-code")

from src.data.dicom import load_series

DATA_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

for _, row in study_series.iterrows():
    plane = row["Anatomical_Plane"]
    series_uid = row["SeriesInstanceUID"]

    series_dir = (
        DATA_ROOT
        / "train_series"
        / str(study_uid)
        / str(series_uid)
    )

    volume, metadata = load_series(series_dir)

    print(
        f"{plane}: "
        f"shape={volume.shape}, "
        f"dtype={volume.dtype}"
    )

In [ ]:
import shutil
from pathlib import Path

CODE_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/rsna-knee-abnormality-code"
)

WORK_ROOT = Path("/kaggle/working/rsna-knee-abnormality-code")

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

shutil.copytree(CODE_ROOT, WORK_ROOT)

print("Code copied successfully.")

In [ ]:
from src.preprocessing.pipeline import preprocess_volume

for _, row in study_series.iterrows():
    plane = row["Anatomical_Plane"]
    series_uid = row["SeriesInstanceUID"]

    series_dir = (
        DATA_ROOT
        / "train_series"
        / str(study_uid)
        / str(series_uid)
    )

    volume, _ = load_series(series_dir)

    processed = preprocess_volume(
        volume,
        num_slices=32,
        image_size=(224, 224),
    )

    print(
        f"{plane}: "
        f"original={volume.shape} → "
        f"processed={tuple(processed.shape)}, "
        f"range=({processed.min():.3f}, {processed.max():.3f})"
    )

In [ ]:
import importlib
import src.models.dinov2 as dinov2_module

importlib.reload(dinov2_module)

print("DINOv2 module reloaded.")

In [ ]:
import torch

from src.models.dinov2 import DINOv2Backbone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DINOv2Backbone(
    model_name="facebook/dinov2-base",
    freeze=True,
).to(device)

model.eval()

# Take the first preprocessed sagittal slice
slice_img = preprocess_volume(
    volume,
    num_slices=32,
    image_size=(224, 224),
)[0]

# (H, W) → (B, C, H, W)
x = slice_img.unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    features = model(x)

print("Input:", x.shape)
print("Features:", features.shape)
print("Feature dimension:", model.feature_dim)

In [ ]:
import importlib
import src.models.dinov2 as dinov2_module

importlib.reload(dinov2_module)

print("DINOv2 module reloaded.")

In [ ]:
import torch

from src.models.dinov2 import DINOv2Backbone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DINOv2Backbone(
    freeze=True,
).to(device)

model.eval()

# Use one real preprocessed MRI slice
slice_img = processed[0]

x = (
    slice_img
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():
    features = model(x)

print("Input:", x.shape)
print("Features:", features.shape)
print("Feature dimension:", model.feature_dim)

In [ ]:
print(type(coronal_volume))
print(len(coronal_volume))
print([type(x) for x in coronal_volume])

In [ ]:
coronal_volume = coronal_volume[0]
sagittal_volume = sagittal_volume[0]
axial_volume = axial_volume[0]

print("Coronal:", coronal_volume.shape)
print("Sagittal:", sagittal_volume.shape)
print("Axial:", axial_volume.shape)

In [ ]:
from src.preprocessing.pipeline import preprocess_volume

processed_coronal = preprocess_volume(coronal_volume)
processed_sagittal = preprocess_volume(sagittal_volume)
processed_axial = preprocess_volume(axial_volume)

print("Coronal:", processed_coronal.shape)
print("Sagittal:", processed_sagittal.shape)
print("Axial:", processed_axial.shape)


In [ ]:
sagittal = processed_sagittal.unsqueeze(0).unsqueeze(2)
coronal = processed_coronal.unsqueeze(0).unsqueeze(2)
axial = processed_axial.unsqueeze(0).unsqueeze(2)

print(sagittal.shape)
print(coronal.shape)
print(axial.shape)

In [ ]:
sagittal = sagittal.to(device)
coronal = coronal.to(device)
axial = axial.to(device)

with torch.no_grad():
    output = model(sagittal, coronal, axial)

print("Output shape:", output.shape)
print("Output:", output)

In [ ]:
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/arjunbhattarai123/rsna-knee-abnormality-code/"
    "src/training/trainer.py"
)

print("Size:", p.stat().st_size)
print("Contains Trainer:", "class Trainer:" in p.read_text())

In [ ]:
import sys
import importlib

# Remove cached src modules
for name in list(sys.modules):
    if name == "src.training.trainer" or name.startswith("src.training"):
        del sys.modules[name]

from src.training.trainer import Trainer

print("Trainer imported successfully.")
print(Trainer)

In [ ]:
from src.data.dataset import KneeDataset

dataset = KneeDataset(
    csv_path="/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv",
    series_csv_path="/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv",
    data_dir="/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series",
)

print("Dataset size:", len(dataset))

sample = dataset[0]

print("Number of outputs:", len(sample))

for i, x in enumerate(sample):
    if hasattr(x, "shape"):
        print(i, x.shape)
    else:
        print(i, type(x))

In [ ]:
from pathlib import Path
import src.data.dataset as dataset_module

p = Path(dataset_module.__file__)

print("Loaded from:", p)
print("File size:", p.stat().st_size)

text = p.read_text()

print("Returns tuple:", "return (" in text)
print("Contains preprocessing:", "preprocess_volume" in text)

In [ ]:
import sys
import importlib

# Remove cached dataset module
for name in list(sys.modules):
    if name == "src.data.dataset":
        del sys.modules[name]

import src.data.dataset as dataset_module
importlib.reload(dataset_module)

KneeDataset = dataset_module.KneeDataset

print("Dataset class loaded from:")
print(dataset_module.__file__)

dataset = KneeDataset(
    csv_path="/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv",
    series_csv_path="/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv",
    data_dir="/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series",
)

sample = dataset[0]

print("Number of outputs:", len(sample))

for i, x in enumerate(sample):
    print(i, type(x), getattr(x, "shape", None))

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

batch = next(iter(loader))

sagittal, coronal, axial, labels = batch

print("Sagittal:", sagittal.shape)
print("Coronal:", coronal.shape)
print("Axial:", axial.shape)
print("Labels:", labels.shape)

In [ ]:
print("Current model:", type(model))
print("Current model.backbone:", type(model.backbone))
print("DINOv2 inside nested model:", type(model.backbone.backbone))

In [ ]:
import inspect

print(inspect.signature(type(model).__init__))

In [ ]:
from src.models.dinov2 import DINOv2Backbone
from src.models.model import KneeModel

# Load the frozen DINOv2 backbone
backbone = DINOv2Backbone(
    model_name="/kaggle/input/models/metaresearch/dinov2/pytorch/base/1",
    freeze=True,
)

# Build the actual knee model
model = KneeModel(
    backbone=backbone,
    feature_dim=backbone.feature_dim,
    num_targets=12,
)

model = model.to(device)

print("Model:", type(model))
print("Backbone:", type(model.backbone))
print("Feature dimension:", model.backbone.feature_dim)

In [ ]:
# Get a fresh batch
batch = next(iter(loader))

sagittal, coronal, axial, labels = batch

sagittal = sagittal.to(device)
coronal = coronal.to(device)
axial = axial.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():
    logits = model(
        sagittal,
        coronal,
        axial,
    )

print("Logits shape:", logits.shape)
print("Labels shape:", labels.shape)

In [ ]:
from src.training.losses import WeightedBCELoss

# Calculate positive weights from the labeled training data
targets = torch.tensor(
    dataset.labels,
    dtype=torch.float32,
)

pos = targets.sum(dim=0)
neg = targets.shape[0] - pos

pos_weight = neg / (pos + 1e-6)

criterion = WeightedBCELoss(
    pos_weight=pos_weight.to(device)
)

model.train()

logits = model(
    sagittal,
    coronal,
    axial,
)

loss = criterion(
    logits,
    labels,
)

loss.backward()

print("Loss:", loss.item())
print("Backward pass: successful")

In [ ]:
!pip install iterative-stratification -q

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/rsna-knee-abnormality-code")

from src.data.splits import MultilabelStratifiedKFold

print("Import successful!")

In [ ]:
from src.data.splits import MultilabelStratifiedKFold
from src.data.dataset import TARGETS
import pandas as pd
import numpy as np

train_csv = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"

df = pd.read_csv(train_csv)

# Keep only fully labeled studies
df = df.dropna(subset=TARGETS).reset_index(drop=True)

print("Labeled studies:", len(df))

X = np.zeros((len(df), 1))
y = df[TARGETS].values.astype(int)

skf = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"val={len(val_idx)}"
    )

In [ ]:
from src.data.dataset import KneeDataset
from src.data.splits import MultilabelStratifiedKFold
import pandas as pd
import numpy as np

TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
SERIES_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

# Dataset
dataset = KneeDataset(
    csv_path=TRAIN_CSV,
    series_csv_path=SERIES_CSV,
    data_dir=DATA_DIR,
    num_slices=32,
    image_size=(224, 224),
)

print("Dataset size:", len(dataset))

# Labels used for stratification
y = dataset.labels.astype(int)
X = np.zeros((len(dataset), 1))

skf = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Create folds
folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    folds.append((train_idx, val_idx))
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"val={len(val_idx)}"
    )

In [ ]:
from torch.utils.data import Subset, DataLoader

train_idx, val_idx = folds[0]

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
from src.models.dinov2 import DINOv2Backbone
from src.models.model import KneeModel

backbone = DINOv2Backbone(
    model_name="/kaggle/input/models/metaresearch/dinov2/pytorch/base/1",
    freeze=True,
)

model = KneeModel(
    backbone=backbone,
    feature_dim=backbone.feature_dim,
    num_targets=12,
).to(device)

print("Model:", type(model))
print("Backbone:", type(model.backbone))
print("Feature dimension:", backbone.feature_dim)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])

In [ ]:
import pandas as pd

train_csv = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
series_csv = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"

train_df = pd.read_csv(train_csv)
train_df = train_df.dropna(subset=TARGETS).reset_index(drop=True)

series_df = pd.read_csv(series_csv)

series_df = series_df[
    (series_df["Fluid_Sensitive"] == 1)
    & (series_df["Anatomical_Plane"].isin(
        ["Sagittal", "Coronal", "Axial"]
    ))
]

planes = ["Sagittal", "Coronal", "Axial"]

for plane in planes:
    studies = set(
        series_df.loc[
            series_df["Anatomical_Plane"] == plane,
            "StudyInstanceUID"
        ].astype(str)
    )

    labeled_studies = set(
        train_df["StudyInstanceUID"].astype(str)
    )

    missing = labeled_studies - studies

    print(f"{plane}: missing in {len(missing)}/58 studies")

print("\nChecking complete 3-view studies...")

complete = 0

for study in train_df["StudyInstanceUID"].astype(str):
    available = set(
        series_df.loc[
            series_df["StudyInstanceUID"] == study,
            "Anatomical_Plane"
        ]
    )

    if all(p in available for p in planes):
        complete += 1

print("Complete 3-view studies:", complete)
print("Incomplete studies:", len(train_df) - complete)

In [ ]:
for study in train_df["StudyInstanceUID"].astype(str):

    available = set(
        series_df.loc[
            series_df["StudyInstanceUID"].astype(str) == study,
            "Anatomical_Plane"
        ]
    )

    missing = [p for p in planes if p not in available]

    if missing:
        print("Study:", study)
        print("Missing:", missing)
        print("Available:", sorted(available))
        print()
        

In [ ]:
from pathlib import Path

dataset_file = Path(
    "/kaggle/working/rsna-knee-abnormality-code/src/data/dataset.py"
)

text = dataset_file.read_text()

old = '''if candidates.empty:
            raise FileNotFoundError(
                f"No fluid-sensitive {plane} series found "
                f"for study {study_uid}"
            )'''

new = '''if candidates.empty:
            return torch.zeros(
                self.num_slices,
                1,
                *self.image_size,
                dtype=torch.float32,
            )'''

if old not in text:
    print("Old code not found — stopping.")
else:
    text = text.replace(old, new)
    dataset_file.write_text(text)
    print("dataset.py updated successfully.")

In [ ]:
import sys
import importlib.util

SRC_PATH = "/kaggle/working/rsna-knee-abnormality-code/src"

spec = importlib.util.spec_from_file_location(
    "src",
    f"{SRC_PATH}/__init__.py",
    submodule_search_locations=[SRC_PATH],
)

src_module = importlib.util.module_from_spec(spec)
sys.modules["src"] = src_module
spec.loader.exec_module(src_module)

print("src registered successfully")

In [ ]:
from pathlib import Path
import importlib
import sys

# Paths
TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
SERIES_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

# Restore src package
SRC_PATH = "/kaggle/working/rsna-knee-abnormality-code/src"

spec = importlib.util.spec_from_file_location(
    "src",
    f"{SRC_PATH}/__init__.py",
    submodule_search_locations=[SRC_PATH],
)

src_module = importlib.util.module_from_spec(spec)
sys.modules["src"] = src_module
spec.loader.exec_module(src_module)

# Load updated dataset
from src.data.dataset import KneeDataset

dataset = KneeDataset(
    csv_path=TRAIN_CSV,
    series_csv_path=SERIES_CSV,
    data_dir=DATA_DIR,
    num_slices=32,
    image_size=(224, 224),
)

# Test the previously failing study
study_uid = "1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132"

idx = dataset.df.index[
    dataset.df["StudyInstanceUID"].astype(str) == study_uid
][0]

sagittal, coronal, axial, labels = dataset[idx]

print("Sagittal:", sagittal.shape)
print("Coronal:", coronal.shape)
print("Axial:", axial.shape)
print("Labels:", labels.shape)
print("Coronal all zeros:", bool((coronal == 0).all()))

In [ ]:
from src.data.splits import MultilabelStratifiedKFold
import numpy as np

y = dataset.labels.astype(int)
X = np.zeros((len(dataset), 1))

skf = MultilabelStratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

folds = list(skf.split(X, y))

train_idx, val_idx = folds[0]

print("Train samples:", len(train_idx))
print("Validation samples:", len(val_idx))

In [ ]:
from torch.utils.data import Subset, DataLoader

train_idx, val_idx = folds[0]

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)

history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=1,
)

In [ ]:
import time

start = time.time()

_ = trainer.train_epoch(train_loader)

elapsed = time.time() - start

print(f"1 training epoch: {elapsed / 60:.2f} minutes")

In [ ]:
# Recreate a fresh DINOv2 backbone
backbone = DINOv2Backbone(
    model_name="/kaggle/input/models/metaresearch/dinov2/pytorch/base/1",
    freeze=True,
)

# Fresh model
model = KneeModel(
    backbone=backbone,
    feature_dim=backbone.feature_dim,
    num_targets=12,
).to(device)

# Fresh optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

print("Fresh model created.")
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)

history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
)

In [ ]:
import pandas as pd

series_df = pd.read_csv(SERIES_CSV)

fs = series_df[
    (series_df["Fluid_Sensitive"] == 1)
    & (series_df["Anatomical_Plane"].isin(
        ["Sagittal", "Coronal", "Axial"]
    ))
].copy()

counts = (
    fs.groupby(["StudyInstanceUID", "Anatomical_Plane"])
      ["SeriesInstanceUID"]
      .nunique()
)

print("Total fluid-sensitive plane groups:", len(counts))
print("\nNumber of series per study/plane:")
print(counts.value_counts().sort_index())

print("\nMaximum series for any study/plane:", counts.max())

In [ ]:
multi = (
    fs.groupby(["StudyInstanceUID", "Anatomical_Plane"])
      .filter(lambda x: len(x) > 1)
)

print(multi.groupby(
    ["Anatomical_Plane", "Fat_Suppression"]
).size())

print("\nUnique combinations:")
print(
    multi[
        ["StudyInstanceUID", "Anatomical_Plane", "Fat_Suppression"]
    ].drop_duplicates()
    .groupby(["Anatomical_Plane", "Fat_Suppression"])
    .size()
)

In [ ]:
multi_counts = (
    fs.groupby(["StudyInstanceUID", "Anatomical_Plane"])
      ["SeriesInstanceUID"]
      .nunique()
)

example = multi_counts[multi_counts >= 2].index[0]

study_uid, plane = example

print("Study:", study_uid)
print("Plane:", plane)

print(
    fs[
        (fs["StudyInstanceUID"] == study_uid) &
        (fs["Anatomical_Plane"] == plane)
    ][["SeriesInstanceUID", "Fluid_Sensitive",
       "Fat_Suppression", "Anatomical_Plane"]]
)

In [ ]:
from pathlib import Path
import pydicom

study_uid = "1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764"

series_uids = (
    fs[
        (fs["StudyInstanceUID"] == study_uid) &
        (fs["Anatomical_Plane"] == "Coronal")
    ]["SeriesInstanceUID"]
    .tolist()
)

root = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series")

for series_uid in series_uids:
    series_dir = root / study_uid / series_uid
    files = list(series_dir.glob("*.dcm"))

    ds = pydicom.dcmread(files[0], stop_before_pixels=True)

    print("\n" + "=" * 60)
    print("Series:", series_uid)
    print("DICOM files:", len(files))
    print("SeriesDescription:", getattr(ds, "SeriesDescription", "N/A"))
    print("ProtocolName:", getattr(ds, "ProtocolName", "N/A"))
    print("SequenceName:", getattr(ds, "SequenceName", "N/A"))
    print("SliceThickness:", getattr(ds, "SliceThickness", "N/A"))
    print("SpacingBetweenSlices:", getattr(ds, "SpacingBetweenSlices", "N/A"))
    print("PixelSpacing:", getattr(ds, "PixelSpacing", "N/A"))

In [ ]:
import pydicom
from collections import Counter

descriptions = []

multi_groups = (
    fs.groupby(["StudyInstanceUID", "Anatomical_Plane"])
      .filter(lambda x: len(x) > 1)
)

for _, row in multi_groups.iterrows():
    study = row["StudyInstanceUID"]
    series = row["SeriesInstanceUID"]

    series_dir = root / study / series
    files = list(series_dir.glob("*.dcm"))

    if not files:
        continue

    ds = pydicom.dcmread(files[0], stop_before_pixels=True)
    descriptions.append(
        (
            row["Anatomical_Plane"],
            getattr(ds, "SeriesDescription", "N/A")
        )
    )

print("Most common duplicate-series descriptions:")
for item, count in Counter(descriptions).most_common(30):
    print(count, "|", item)

In [ ]:
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"

df = pd.read_csv(TRAIN_CSV)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

df = df.dropna(subset=TARGETS).reset_index(drop=True)

print("Labeled studies:", len(df))

In [ ]:
labeled_ids = set(df["StudyInstanceUID"])

labeled_fs = fs[
    fs["StudyInstanceUID"].isin(labeled_ids)
].copy()

print("Labeled fluid-sensitive series:", len(labeled_fs))

print("\nSeries count per labeled study/plane:")
print(
    labeled_fs.groupby(
        ["StudyInstanceUID", "Anatomical_Plane"]
    )["SeriesInstanceUID"]
    .nunique()
    .value_counts()
    .sort_index()
)

In [ ]:
# Show every fluid-sensitive candidate for the 58 labeled studies
for study_uid in sorted(labeled_ids):
    print("\n" + "=" * 80)
    print("STUDY:", study_uid)

    rows = labeled_fs[
        labeled_fs["StudyInstanceUID"] == study_uid
    ]

    for _, row in rows.iterrows():
        series_dir = root / study_uid / row["SeriesInstanceUID"]
        files = list(series_dir.glob("*.dcm"))

        if not files:
            continue

        ds = pydicom.dcmread(files[0], stop_before_pixels=True)

        print(
            f"{row['Anatomical_Plane']:9s} | "
            f"{len(files):2d} slices | "
            f"{getattr(ds, 'SeriesDescription', 'N/A')}"
        )

In [ ]:
import re

def classify_description(desc):
    desc = str(desc).lower()

    if "acl" in desc or "pcl" in desc:
        return "TARGETED_LIGAMENT"

    if "pd" in desc or "pdw" in desc:
        return "GENERAL_PD"

    if "t2" in desc or "stir" in desc:
        return "GENERAL_T2"

    if "dummy" in desc or desc == "n/a":
        return "UNKNOWN"

    return "OTHER"


results = []

for study_uid in sorted(labeled_ids):
    rows = labeled_fs[
        labeled_fs["StudyInstanceUID"] == study_uid
    ]

    for plane in ["Sagittal", "Coronal", "Axial"]:
        candidates = rows[
            rows["Anatomical_Plane"] == plane
        ]

        if len(candidates) <= 1:
            continue

        for _, row in candidates.iterrows():
            series_dir = root / study_uid / row["SeriesInstanceUID"]
            files = list(series_dir.glob("*.dcm"))

            if not files:
                continue

            ds = pydicom.dcmread(files[0], stop_before_pixels=True)
            desc = getattr(ds, "SeriesDescription", "N/A")

            results.append({
                "study": study_uid,
                "plane": plane,
                "series": row["SeriesInstanceUID"],
                "slices": len(files),
                "description": desc,
                "class": classify_description(desc),
            })

results_df = pd.DataFrame(results)

print(
    results_df.groupby(["plane", "class"])
    .size()
    .sort_values(ascending=False)
)

print("\nTARGETED sequences:")
print(
    results_df[
        results_df["class"] == "TARGETED_LIGAMENT"
    ][["plane", "slices", "description"]]
    .to_string(index=False)
)

In [2]:
import sys
import importlib.util

SRC_PATH = "/kaggle/working/rsna-knee-abnormality-code/src"

spec = importlib.util.spec_from_file_location(
    "src",
    f"{SRC_PATH}/__init__.py",
    submodule_search_locations=[SRC_PATH],
)

src_module = importlib.util.module_from_spec(spec)
sys.modules["src"] = src_module
spec.loader.exec_module(src_module)

from src.data.dataset import KneeDataset

In [6]:
BASE_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

TRAIN_CSV_PATH = f"{BASE_DIR}/train.csv"
SERIES_CSV_PATH = f"{BASE_DIR}/train_series.csv"
DATA_DIR = f"{BASE_DIR}/train_series"

In [7]:
import os

sample_study = os.listdir(DATA_DIR)[0]
print("study folder:", sample_study)

sample_series = os.listdir(f"{DATA_DIR}/{sample_study}")[0]
print("series folder:", sample_series)

print(os.listdir(f"{DATA_DIR}/{sample_study}/{sample_series}")[:5])

study folder: 1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839
series folder: 1.2.826.0.1.3680043.8.498.28798546215369442215420222435134673142
['1.2.826.0.1.3680043.8.498.12486678153168904030743072482872142065.dcm', '1.2.826.0.1.3680043.8.498.86700687663805155070115680323395754377.dcm', '1.2.826.0.1.3680043.8.498.40362622123295797247686695535473040667.dcm', '1.2.826.0.1.3680043.8.498.22606964411658655718893941683908738828.dcm', '1.2.826.0.1.3680043.8.498.47226698003996911677858077867948563321.dcm']


In [8]:
BASE_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

TRAIN_CSV_PATH = f"{BASE_DIR}/train.csv"
SERIES_CSV_PATH = f"{BASE_DIR}/train_series.csv"
DATA_DIR = f"{BASE_DIR}/train_series"

import pandas as pd

dataset = KneeDataset(
    csv_path=TRAIN_CSV_PATH,
    series_csv_path=SERIES_CSV_PATH,
    data_dir=DATA_DIR,
)

rows = []
for study_uid in dataset.df["StudyInstanceUID"].astype(str):
    for plane in ["Sagittal", "Coronal", "Axial"]:
        series_uid = dataset._select_best_series_uid(study_uid, plane)

        desc = None
        if series_uid is not None:
            series_dir = dataset.data_dir / study_uid / series_uid
            dcm_files = list(series_dir.glob("*.dcm"))
            if dcm_files:
                import pydicom
                ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
                desc = getattr(ds, "SeriesDescription", None)

        rows.append({
            "study_uid": study_uid,
            "plane": plane,
            "series_uid": series_uid,
            "series_description": desc,
        })

selection_df = pd.DataFrame(rows)
selection_df.to_csv("series_selection_report.csv", index=False)

print(selection_df["plane"].value_counts())
print("\nMissing per plane:")
print(selection_df[selection_df["series_uid"].isna()]["plane"].value_counts())

print("\nSample selections:")
print(selection_df.head(20))

plane
Sagittal    58
Coronal     58
Axial       58
Name: count, dtype: int64

Missing per plane:
plane
Coronal     2
Sagittal    2
Name: count, dtype: int64

Sample selections:
                                            study_uid     plane  \
0   1.2.826.0.1.3680043.8.498.10095687747295410396...  Sagittal   
1   1.2.826.0.1.3680043.8.498.10095687747295410396...   Coronal   
2   1.2.826.0.1.3680043.8.498.10095687747295410396...     Axial   
3   1.2.826.0.1.3680043.8.498.10170898615867673028...  Sagittal   
4   1.2.826.0.1.3680043.8.498.10170898615867673028...   Coronal   
5   1.2.826.0.1.3680043.8.498.10170898615867673028...     Axial   
6   1.2.826.0.1.3680043.8.498.10306159113324811538...  Sagittal   
7   1.2.826.0.1.3680043.8.498.10306159113324811538...   Coronal   
8   1.2.826.0.1.3680043.8.498.10306159113324811538...     Axial   
9   1.2.826.0.1.3680043.8.498.11287937729196958426...  Sagittal   
10  1.2.826.0.1.3680043.8.498.11287937729196958426...   Coronal   
11  1.2.826.0.1.368

In [9]:
target_study = "1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764"
print(selection_df[selection_df["study_uid"] == target_study])

Empty DataFrame
Columns: [study_uid, plane, series_uid, series_description]
Index: []


In [10]:
targeted_selected = selection_df[
    selection_df["series_description"].str.upper().str.contains("ACL|PCL|OBL", na=False)
]
print(f"Targeted sequences selected: {len(targeted_selected)}")
print(targeted_selected)

Targeted sequences selected: 0
Empty DataFrame
Columns: [study_uid, plane, series_uid, series_description]
Index: []


In [11]:
target_study = "1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764"
print(target_study in dataset.df["StudyInstanceUID"].astype(str).values)

False
